# Module 01 — Le mur de la mémoire

**Formation Big Data — ANSD / Data Innovation Lab**

Au notebook précédent, tout fonctionnait : 100 000 lignes, quelques dixièmes de
seconde. Nous chargeons maintenant le **fichier complet**.

Objectifs :

1. vérifier l'estimation de mémoire que vous avez posée au notebook 01 ;
2. observer où part la mémoire, et pourquoi le fichier occupe plus de place en
   RAM que sur le disque ;
3. constater le second mur, moins visible : **un seul cœur travaille** ;
4. mesurer ce que coûtent réellement un tri et une jointure.

---

> ⚠️ **Ce notebook est conçu pour aller trop loin.** Il est normal, et même
> souhaitable, que votre noyau (*kernel*) meure en cours de route. Si cela
> arrive : redémarrez le noyau, réduisez `N_LIGNES` dans la cellule prévue,
> et reprenez. Le constat reste valable à toutes les échelles.

## 1. Préparation et outils de mesure

In [ ]:
import gc
import os
import threading
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

DOSSIER_DONNEES = Path("..") / "00-data"
FICHIER = DOSSIER_DONNEES / "individus.csv"

processus = psutil.Process(os.getpid())


def memoire_processus_mo():
    """Mémoire réellement occupée par ce noyau Python, en Mo."""
    return processus.memory_info().rss / 1024**2


def memoire_libre_go():
    return psutil.virtual_memory().available / 1024**3


def pic_memoire(fonction, intervalle=0.02):
    """Exécute `fonction` en surveillant le pic de mémoire du noyau.

    Renvoie (résultat, durée en s, pic de mémoire en Mo).
    """
    arret = False
    sommet = [memoire_processus_mo()]

    def surveiller():
        while not arret:
            sommet[0] = max(sommet[0], memoire_processus_mo())
            time.sleep(intervalle)

    veilleur = threading.Thread(target=surveiller, daemon=True)
    veilleur.start()
    depart = time.perf_counter()
    try:
        resultat = fonction()
    finally:
        arret = True
        veilleur.join()
    return resultat, time.perf_counter() - depart, sommet[0]


taille_disque_mo = FICHIER.stat().st_size / 1024**2
print(f"Fichier          : {taille_disque_mo:,.0f} Mo sur le disque".replace(",", " "))
print(f"Mémoire libre    : {memoire_libre_go():.1f} Go")
print(f"Cœurs logiques   : {os.cpu_count()}")
print(f"Noyau Python     : {memoire_processus_mo():.0f} Mo occupés")

## 2. Votre estimation, avant de charger

Au notebook 01, l'échantillon de 100 000 lignes occupait une certaine place en
mémoire. Extrapolons.

In [ ]:
# Estimation de la mémoire nécessaire pour le fichier complet.
# On compte d'abord les lignes SANS charger le fichier : on le parcourt
# ligne à ligne, ce qui ne consomme presque rien.

with FICHIER.open("rb") as fichier:
    nb_lignes_total = sum(1 for _ in fichier) - 1        # -1 pour l'en-tête

# Mémoire des 100 000 lignes de l'échantillon, en Mo
echantillon = pd.read_csv(FICHIER, nrows=100_000)
memoire_echantillon_mo = echantillon.memory_usage(deep=True).sum() / 1024**2
del echantillon
gc.collect()

estimation_go = memoire_echantillon_mo * (nb_lignes_total / 100_000) / 1024

print(f"Lignes dans le fichier   : {nb_lignes_total:,}".replace(",", " "))
print(f"Mémoire pour 100 000     : {memoire_echantillon_mo:.0f} Mo")
print(f"Estimation : {estimation_go:.2f} Go pour {nb_lignes_total:,} lignes"
      .replace(",", " "))

**Question 1.** Comparez votre estimation à la taille du fichier sur le
disque. Attendez-vous à ce que la mémoire soit *plus* ou *moins* importante que
le disque ? Pourquoi ?

*Votre réponse :* …

## 3. Le chargement

`N_LIGNES = None` charge la totalité du fichier. **Si votre noyau meurt**,
remplacez par une valeur (par exemple `3_000_000`), redémarrez et relancez.

In [ ]:
N_LIGNES = 6_000_000   # ← à réduire si le noyau ne survit pas

avant = memoire_processus_mo()
print(f"Avant chargement : {avant:.0f} Mo")

df, duree, pic = pic_memoire(lambda: pd.read_csv(FICHIER, nrows=N_LIGNES))
gc.collect()

apres = memoire_processus_mo()
taille_df_mo = df.memory_usage(deep=True).sum() / 1024**2

print(f"Durée de lecture : {duree:,.1f} s".replace(",", " "))
print(f"Lignes chargées  : {len(df):,}".replace(",", " "))
print(f"Table en mémoire : {taille_df_mo:,.0f} Mo".replace(",", " "))
print(f"Noyau après      : {apres:,.0f} Mo".replace(",", " "))
print(f"PIC pendant la lecture : {pic:,.0f} Mo".replace(",", " "))

**Le premier enseignement est là.** Regardez l'écart entre le pic atteint
*pendant* la lecture et la place occupée *après*.

Pour lire un fichier, pandas assemble les blocs lus puis les recopie dans la
table finale : à l'instant le plus critique, les deux coexistent. Il ne suffit
donc pas que le fichier tienne en mémoire — il faut environ **deux fois** sa
place pour parvenir à le charger.

**Question 2.** Quel rapport observez-vous entre la taille sur le disque et la
taille en mémoire ? Et entre le pic et la taille finale ?

*Votre réponse :* …

## 4. Où part la mémoire ?

In [ ]:
(df.memory_usage(deep=True) / 1024**2).round(1).sort_values(ascending=False)

In [ ]:
df.dtypes.value_counts()

Selon votre version de pandas, les colonnes de texte sont stockées
différemment : jusqu'à pandas 2, chaque chaîne était un objet Python isolé,
très coûteux ; depuis pandas 3, un format compact (Arrow) est utilisé par
défaut. Le rapport disque → mémoire s'en trouve nettement réduit… mais le mur
reste : **tout le fichier atterrit en mémoire d'un seul bloc**.

### Première parade : les types catégoriels

Beaucoup de nos colonnes ne prennent qu'un petit nombre de valeurs distinctes
(14 régions, 2 sexes, 6 niveaux d'instruction…). Les stocker comme des
catégories revient à conserver un dictionnaire des valeurs et, pour chaque
ligne, un simple numéro.

In [ ]:
# Fourni : nombre de valeurs distinctes par colonne texte
colonnes_texte = [col for col in df.columns if df[col].dtype == object
                  or str(df[col].dtype) in ("str", "string")]
df[colonnes_texte].nunique().sort_values()

In [ ]:
# Conversion en `category` des colonnes à faible cardinalité.
#
# On écarte `id_menage` et `date_naissance` : leurs valeurs sont presque
# toutes différentes, donc le dictionnaire serait aussi gros que la colonne.

cardinalites = df[colonnes_texte].nunique()
colonnes_a_convertir = [c for c in cardinalites[cardinalites < 100].index
                        if c not in ("id_menage", "date_naissance")]

print("Colonnes converties :", colonnes_a_convertir)

df_compact = df.copy()
for colonne in colonnes_a_convertir:
    df_compact[colonne] = df_compact[colonne].astype("category")

taille_compacte_mo = df_compact.memory_usage(deep=True).sum() / 1024**2
gain = 100 * (1 - taille_compacte_mo / taille_df_mo)
print(f"\nAvant : {taille_df_mo:,.0f} Mo".replace(",", " "))
print(f"Après : {taille_compacte_mo:,.0f} Mo  (gain {gain:.0f} %)".replace(",", " "))

In [ ]:
del df_compact
gc.collect()
print(f"Noyau : {memoire_processus_mo():,.0f} Mo".replace(",", " "))

### Deuxième parade : ne lire que ce dont on a besoin

Si votre analyse ne porte que sur cinq colonnes, pourquoi en charger vingt et
une ?

In [ ]:
# Relecture en ne lisant que les colonnes nécessaires au taux d'activité
colonnes_utiles = ["region", "milieu_residence", "age", "situation_activite"]

df_partiel, duree_partielle, pic_partiel = pic_memoire(
    lambda: pd.read_csv(FICHIER, usecols=colonnes_utiles, nrows=N_LIGNES)
)

print(f"Durée : {duree_partielle:,.1f} s (contre {duree:,.1f} s)".replace(",", " "))
print(f"Pic   : {pic_partiel:,.0f} Mo".replace(",", " "))
print(f"Table : {df_partiel.memory_usage(deep=True).sum() / 1024**2:,.0f} Mo"
      .replace(",", " "))

In [ ]:
del df_partiel
gc.collect()

Ces deux parades sont utiles et vous les emploierez au quotidien. Mais
observez ce qu'elles font : elles **repoussent** le mur, elles ne le
suppriment pas. Doublez le volume de données, et vous y revenez.

## 5. Le second mur : un seul cœur

Votre machine possède plusieurs cœurs. Pendant que le calcul suivant tourne,
ouvrez le moniteur système de votre poste (Gestionnaire des tâches sous
Windows, Moniteur d'activité sous macOS, `htop` sous Linux ou en terminal) et observez
l'utilisation des processeurs.

In [ ]:
print(f"Cœurs disponibles : {os.cpu_count()}")

# Un calcul volontairement long : regardez vos cœurs pendant l'exécution
resultat, duree_tri, pic_tri = pic_memoire(
    lambda: df.sort_values(["region", "age"])
)
print(f"Tri effectué en {duree_tri:,.1f} s".replace(",", " "))
del resultat
gc.collect()

**Question 3.** Combien de cœurs ont travaillé ? Que pourriez-vous faire
des autres ?

*Votre réponse :* …

C'est le second mur : pandas exécute presque tout **séquentiellement, sur un
seul cœur**. Une machine à 8 cœurs travaille à un huitième de ses capacités.

## 6. Le coup de grâce : le coût caché des opérations

Charger la table n'est que le début. Voyons ce que coûte un simple tri.

In [ ]:
# Pic de mémoire pendant un tri, puis pendant une jointure

base = memoire_processus_mo()
print(f"Point de départ : {base:,.0f} Mo".replace(",", " "))

reference = df[["id_individu", "age"]].rename(columns={"age": "age_bis"})

_, duree_a, pic_a = pic_memoire(lambda: df.sort_values("age"))
print(f"Tri      : {duree_a:5.1f} s | pic {pic_a:,.0f} Mo "
      f"(+{pic_a - base:,.0f})".replace(",", " "))

_, duree_b, pic_b = pic_memoire(
    lambda: df.merge(reference, on="id_individu", how="left"))
print(f"Jointure : {duree_b:5.1f} s | pic {pic_b:,.0f} Mo "
      f"(+{pic_b - base:,.0f})".replace(",", " "))

print(f"\nPour mémoire, la table occupe {taille_df_mo:,.0f} Mo."
      .replace(",", " "))

**Question 4.** De combien la mémoire augmente-t-elle pendant ces
opérations, rapporté à la taille de la table ? Qu'en concluez-vous sur la place
libre nécessaire pour travailler sur un fichier donné ?

*Votre réponse :* …

## 7. Synthèse — les deux murs

Complétez ce tableau récapitulatif avec vos propres mesures :

| Grandeur | Votre mesure |
|---|---|
| Taille du fichier sur le disque | … Mo |
| Lignes chargées | … |
| Table en mémoire | … Mo |
| Pic pendant la lecture | … Mo |
| Gain avec les types catégoriels | … % |
| Surcoût d'un tri | … Mo |
| Durée de la lecture | … s |
| Cœurs utilisés | 1 sur … |

Deux obstacles, et un seul d'entre eux se contourne par l'optimisation :

**Mur n° 1 — la mémoire.** Tout le fichier doit tenir en RAM, et il faut de la
place en plus pour le lire, le trier, le joindre. Les types catégoriels et la
lecture partielle repoussent l'échéance ; ils ne changent pas la nature du
problème.

**Mur n° 2 — le mono-cœur.** Un seul processeur travaille, quelle que soit la
machine. Aucune optimisation de types n'y changera quoi que ce soit.

➡️ Notebook suivant : `03_echelle_de_la_douleur.ipynb`. Nous
mesurons ce que ça fait de passer à l'échelle (données de plus en plus grosses).

In [ ]:
# Libération de la mémoire avant de passer à la suite
del df
gc.collect()
print(f"Noyau : {memoire_processus_mo():,.0f} Mo".replace(",", " "))